MSA 驗證階段

In [1]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm


# ==================================================
# 1. 路徑與讀取
# ==================================================

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "content"
FIG_DIR = ROOT / "content" / "figures"
RESULT_DIR = ROOT / "content" / "results"

FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_DIR / "msa_data.csv")

required_columns = [
    "wafer_id", "operator_id", "measured_cd_nm"
]
df = df[required_columns]

df.head()

,wafer_id,operator_id,measured_cd_nm
0,W001,OP01,45.349301
1,W001,OP01,45.337109
2,W001,OP02,45.331697
3,W001,OP02,45.459043
4,W001,OP03,45.239875


In [2]:
# ==================================================
# 2. Crossed ANOVA：
# ==================================================
model = ols(
    "measured_cd_nm ~ C(wafer_id) * C(operator_id)",
    data=df ).fit()

anova_table = anova_lm(model, typ=2)

print("\n========== Crossed ANOVA ==========")
print(anova_table)

anova_table.to_csv(
    RESULT_DIR / "msa_anova_table.csv"
)

# SS、DF、MS
ss_part = anova_table.loc["C(wafer_id)", "sum_sq"]
df_part = anova_table.loc["C(wafer_id)", "df"]

ss_operator = anova_table.loc["C(operator_id)", "sum_sq"]
df_operator = anova_table.loc["C(operator_id)", "df"]

ss_interaction = anova_table.loc[
    "C(wafer_id):C(operator_id)", "sum_sq"
]
df_interaction = anova_table.loc[
    "C(wafer_id):C(operator_id)", "df"
]

ss_repeat = anova_table.loc["Residual", "sum_sq"]
df_repeat = anova_table.loc["Residual", "df"]

ms_part = ss_part / df_part
ms_operator = ss_operator / df_operator
ms_interaction = ss_interaction / df_interaction
ms_repeat = ss_repeat / df_repeat

print("\n========== Mean Squares ==========")
print(f"MS Part        : {ms_part:.6f}")
print(f"MS Operator    : {ms_operator:.6f}")
print(f"MS Interaction : {ms_interaction:.6f}")
print(f"MS Repeat      : {ms_repeat:.6f}")



========== Crossed ANOVA ==========
                               sum_sq    df           F        PR(>F)
C(wafer_id)                 30.724879   9.0  474.926580  7.924543e-30
C(operator_id)               0.151964   2.0   10.570352  3.352110e-04
C(wafer_id):C(operator_id)   0.177320  18.0    1.370450  2.167796e-01
Residual                     0.215647  30.0         NaN           NaN

========== Mean Squares ==========
MS Part        : 3.413875
MS Operator    : 0.075982
MS Interaction : 0.009851
MS Repeat      : 0.007188


In [3]:
# ==================================================
# 3. Interaction Pooling
# ==================================================

n_parts = df["wafer_id"].nunique()
n_operators = df["operator_id"].nunique()

# 每個 Part × Operator 組合的重複測量次數
repeats_per_cell = (
    df.groupby(["wafer_id", "operator_id"])
    .size()
)

if repeats_per_cell.nunique() != 1:
    raise ValueError(
        "目前資料不是平衡設計，不能直接使用這份簡化的 pooling 計算。"
    )

n_repeats = int(repeats_per_cell.iloc[0])

# 將 Interaction SS 與 Repeatability SS 合併
ss_pooled_error = ss_interaction + ss_repeat
df_pooled_error = df_interaction + df_repeat
ms_pooled_error = ss_pooled_error / df_pooled_error

# Pooled model 的變異分量
var_repeat_pooled = ms_pooled_error

var_operator_pooled = max(
    0,
    (ms_operator - ms_pooled_error)
    / (n_parts * n_repeats)
)

var_part_pooled = max(
    0,
    (ms_part - ms_pooled_error)
    / (n_operators * n_repeats)
)

var_reproducibility_pooled = var_operator_pooled

var_grr_pooled = (
    var_repeat_pooled
    + var_reproducibility_pooled
)

var_total_pooled = (
    var_grr_pooled
    + var_part_pooled
)

sd_grr_pooled = np.sqrt(var_grr_pooled)
sd_total_pooled = np.sqrt(var_total_pooled)

study_variation_grr = 6 * sd_grr_pooled
study_variation_total = 6 * sd_total_pooled

pct_contribution_grr = (
    var_grr_pooled / var_total_pooled * 100
)

pct_study_variation_grr = (
    sd_grr_pooled / sd_total_pooled * 100
)

ndc_pooled = np.floor(
    1.41 * np.sqrt(var_part_pooled / var_grr_pooled)
)

pooled_results = pd.DataFrame({
    "Metric": [
        "Repeatability Variance",
        "Operator Variance",
        "Reproducibility Variance",
        "Total GRR Variance",
        "Part-to-Part Variance",
        "Total Variance",
        "GRR Study Variation (6SD)",
        "Total Study Variation (6SD)",
        "GRR % Contribution",
        "GRR % Study Variation",
        "ndc"
    ],
    "Value": [
        var_repeat_pooled,
        var_operator_pooled,
        var_reproducibility_pooled,
        var_grr_pooled,
        var_part_pooled,
        var_total_pooled,
        study_variation_grr,
        study_variation_total,
        pct_contribution_grr,
        pct_study_variation_grr,
        ndc_pooled
    ]
})

print("\n========== Interaction Pooling ==========")
print(pooled_results.to_string(index=False))

pooled_results.to_csv(
    RESULT_DIR / "msa_pooled_results.csv",
    index=False
)



========== Interaction Pooling ==========
                     Metric     Value
     Repeatability Variance  0.008187
          Operator Variance  0.003390
   Reproducibility Variance  0.003390
         Total GRR Variance  0.011577
      Part-to-Part Variance  0.567615
             Total Variance  0.579191
  GRR Study Variation (6SD)  0.645566
Total Study Variation (6SD)  4.566277
         GRR % Contribution  1.998744
      GRR % Study Variation 14.137696
                        ndc  9.000000


In [4]:
# ==================================================
# 4. Operator Post-hoc：配對比較
# ==================================================
# 先將同一個 Wafer、同一個 Operator 的兩次量測取平均
cell_means = (
    df.groupby(["wafer_id", "operator_id"])["measured_cd_nm"]
    .mean()
    .unstack("operator_id")
)

print("\n========== Operator Cell Means ==========")
print(cell_means)

operators = list(cell_means.columns)

pairwise_results = []

for op1, op2 in combinations(operators, 2):

    # 同一批 Wafer 由兩位 Operator 量測
    # 使用 paired t-test 保留 Wafer 配對關係
    t_stat, p_value = stats.ttest_rel(
        cell_means[op1],
        cell_means[op2]
    )

    mean_difference = (
        cell_means[op1] - cell_means[op2]
    ).mean()

    pairwise_results.append({
        "Operator_1": op1,
        "Operator_2": op2,
        "Mean_Difference_nm": mean_difference,
        "t_statistic": t_stat,
        "p_raw": p_value
    })

pairwise_df = pd.DataFrame(pairwise_results)


# Holm correction：修正多重比較
p_values = pairwise_df["p_raw"].to_numpy()
m = len(p_values)

order = np.argsort(p_values)
adjusted = np.empty(m)

for rank, idx in enumerate(order):
    adjusted[idx] = min(
        1.0,
        (m - rank) * p_values[idx]
    )

# 確保調整後 p-value 單調不遞減
for i in range(1, m):
    adjusted[order[i]] = max(
        adjusted[order[i]],
        adjusted[order[i - 1]]
    )

pairwise_df["p_holm"] = adjusted
pairwise_df["Significant"] = (
    pairwise_df["p_holm"] < 0.05
)

print("\n========== Operator Paired Comparisons ==========")
print(pairwise_df.to_string(index=False))

pairwise_df.to_csv(
    RESULT_DIR / "msa_operator_posthoc.csv",
    index=False
)



========== Operator Cell Means ==========
operator_id       OP01       OP02       OP03
wafer_id                                    
W001         45.343205  45.395370  45.160267
W002         44.132605  44.297724  44.056065
W003         45.664442  45.633533  45.594226
W004         45.803065  45.936566  45.612891
W005         43.543869  43.461926  43.368739
W006         44.035441  44.012255  43.918379
W007         45.167974  45.227062  45.097497
W008         44.640396  44.760451  44.760194
W009         44.992706  44.945494  44.971498
W010         44.407839  44.348900  44.297565

========== Operator Paired Comparisons ==========
Operator_1 Operator_2  Mean_Difference_nm  t_statistic    p_raw   p_holm  Significant
      OP01       OP02           -0.028774    -1.021982 0.333473 0.333473        False
      OP01       OP03            0.089422     3.061943 0.013529 0.029235         True
      OP02       OP03            0.118196     3.265979 0.009745 0.029235         True


In [5]:
# ==================================================
# 5. Residual Diagnostics
# ==================================================

residuals = model.resid
fitted = model.fittedvalues

# 5-1. Residual vs Fitted
plt.figure(figsize=(7, 5))

plt.scatter(fitted, residuals, alpha=0.7)
plt.axhline(0, linestyle="--")

plt.xlabel("Fitted CD (nm)")
plt.ylabel("Residual (nm)")
plt.title("Residuals vs Fitted Values")

plt.tight_layout()
plt.savefig(
    FIG_DIR / "msa_residual_vs_fitted.png",
    dpi=300
)
plt.close()


# 5-2. Residual Histogram
plt.figure(figsize=(7, 5))

plt.hist(residuals, bins=12, edgecolor="black")

plt.xlabel("Residual (nm)")
plt.ylabel("Frequency")
plt.title("Residual Distribution")

plt.tight_layout()
plt.savefig(
    FIG_DIR / "msa_residual_histogram.png",
    dpi=300
)
plt.close()


# 5-3. Normal Q-Q Plot
plt.figure(figsize=(7, 5))

stats.probplot(
    residuals,
    dist="norm",
    plot=plt
)

plt.title("Normal Q-Q Plot")

plt.tight_layout()
plt.savefig(
    FIG_DIR / "msa_residual_qq.png",
    dpi=300
)
plt.close()


# 5-4. Residual by Operator
residual_df = df[["operator_id"]].copy()
residual_df["residual"] = residuals

plt.figure(figsize=(7, 5))

residual_df.boxplot(
    column="residual",
    by="operator_id"
)

plt.axhline(0, linestyle="--")
plt.xlabel("Operator")
plt.ylabel("Residual (nm)")
plt.title("Residuals by Operator")
plt.suptitle("")

plt.tight_layout()
plt.savefig(
    FIG_DIR / "msa_residual_by_operator.png",
    dpi=300
)
plt.close()


# 5-5. Shapiro-Wilk normality test
shapiro_stat, shapiro_p = stats.shapiro(residuals)

print("\n========== Residual Normality Test ==========")
print(f"Shapiro-Wilk statistic: {shapiro_stat:.6f}")
print(f"p-value              : {shapiro_p:.6f}")


print("結果：", RESULT_DIR)
print("圖表：", FIG_DIR)


========== Residual Normality Test ==========
Shapiro-Wilk statistic: 0.992237
p-value              : 0.968530
結果： /content/results
圖表： /content/figures


<Figure size 700x500 with 0 Axes>

In [6]:
from scipy.stats import f

# Crossed Random-Effects ANOVA
F_part = ms_part / ms_interaction
F_operator = ms_operator / ms_interaction
F_interaction = ms_interaction / ms_repeat

p_part = f.sf(F_part, df_part, df_interaction)
p_operator = f.sf(F_operator, df_operator, df_interaction)
p_interaction = f.sf(
    F_interaction,
    df_interaction,
    df_repeat
)

corrected_anova = pd.DataFrame({
    "Source": ["Part", "Operator", "Part x Operator"],
    "F": [F_part, F_operator, F_interaction],
    "p_value": [p_part, p_operator, p_interaction]
})

print("\n========== Corrected Crossed ANOVA ==========")
print(corrected_anova.to_string(index=False))


========== Corrected Crossed ANOVA ==========
         Source          F      p_value
           Part 346.548016 2.683881e-18
       Operator   7.713055 3.807852e-03
Part x Operator   1.370450 2.167796e-01


**MSA 結果解讀**

**1.Interaction Pooling：結果穩定**
- GRR 從 14.29% 降至 14.14%，差異非常小。
- ndc 維持 9，量測系統區分不同 wafer 的能力沒有實質改變。
- 交互作用不顯著（p = 0.2168），Pooling 是合理的候選模型。

*本次MSA的主要問題不是不同操作員對不同 wafer 的反應模式不同，而是操作員之間存在平均量測值差異。*

不過，這是依目前模擬設計與樣本得出的結果，不代表真實產線。

**2.Operator Post-hoc：OP03 是主要差異來源**

- OP01 與 OP02：沒有足夠證據認為兩者平均量測值不同。
- OP01 與 OP03：OP01 平均比 OP03 高約 0.089 nm。
- OP02 與 OP03：OP02 平均比 OP03 高約 0.118 nm。

OP03 的平均量測值顯著偏低。
在半導體產線，假設三位操作員使用相同的量測設備、相同的 recipe，測量同一批 wafer：
- OP01、OP02 的結果相對接近。
- OP03 的結果系統性偏低。
- 如果沒有監控操作員差異，可能把人員造成的量測偏差誤認為製程 CD 真的下降。

實務上可能需要檢查：
1. 操作員的量測定位方式是否一致。
2. 取樣位置、對焦或量測區域是否一致。
3. Recipe 的執行流程是否有差異。
4. 是否需要重新訓練或建立標準操作程序（SOP）。
但目前只能說 OP03 的平均結果較低，不能直接認定是OP03操作錯誤。

**3.Residual Diagnostics：**

目前因為 p = 0.9685，大於 0.05，所以沒有足夠證據拒絕殘差符合常態分布的假設。

- 但這不代表已經證明殘差服從常態分布。
- Shapiro-Wilk 只檢查常態性。
- 還需要看 Residual vs Fitted，確認是否有趨勢、異質變異或特殊模式。

因此，這部分暫時判定為「常態性檢查未發現明顯問題」，還不能說模型診斷全部通過。

**目前判讀**
- 量測系統變異相對於總變異占比不高。
- ndc = 9，具有一定區分 wafer 差異的能力，但尚未達到常見的 10 以上參考門檻。
- Operator 效果顯著，OP03 與另外兩位操作員有平均量測差異。
- Part × Operator 交互作用未達顯著。
- 殘差常態性檢查未發現明顯問題。